In [14]:
import json
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

In [15]:
load_dotenv(override=True)

True

In [4]:
client = OpenAI()

In [16]:
def currency_converter(amount, from_currency, to_currency):
    if from_currency == 'USD' and to_currency == 'KGS':
        return {
            "type": "currency_converter",
            "amount": amount,
            "from_currency": from_currency,
            "to_currency": to_currency,
            "result": amount * 84  # Assuming 1 USD = 84 KGS
        }
    if from_currency == 'KGS' and to_currency == 'USD':
        return {
            "type": "currency_converter",
            "amount": amount,
            "from_currency": from_currency,
            "to_currency": to_currency,
            "result": amount / 84  # Assuming 1 USD = 84 KGS
        }
    return {
        "error": "Unsupported currency conversion",
        "description": "Only USD to KGS and KGS to USD conversions are supported"
    }

In [17]:
display(Markdown(json.dumps(currency_converter(1000, "KGS", "USD"))))

{"type": "currency_converter", "amount": 1000, "from_currency": "KGS", "to_currency": "USD", "result": 11.904761904761905}

In [18]:
tools = [{
    "type": "function",
    "function": {
        "name": "currency_converter",
        "description": "Converts currency between USD and KGS, or KGS and USD",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "The amount of money to convert"
                },
                "from_currency": {
                    "type": "string",
                    "description": "The currency to convert from (USD or KGS)"
                },
                "to_currency": {
                    "type": "string",
                    "description": "The currency to convert to (USD or KGS)"
                }
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    }
}]

In [19]:
system_prompt = """
You are banking expert knowledgeable in currency conversion between USD and KGS.
Currency conversion is between Kyrgyzstani Som (KGS) and US Dollars (USD), or vice versa.
"""

In [26]:
def bank_expert(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(model="gpt-5.4-nano", messages=messages, tools=tools)
    
    assistant_response = response.choices[0].message
    messages.append(assistant_response)
    
    if assistant_response.tool_calls:
        for tool_call in assistant_response.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)
            
            if fn_name == "currency_converter":
                result = currency_converter(
                    amount=fn_args["amount"],
                    from_currency=fn_args["from_currency"],
                    to_currency=fn_args["to_currency"]
                )
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

        final_response = client.chat.completions.create(model="gpt-5.4-nano", messages=messages, tools=tools)
        return final_response.choices[0].message.content
    return assistant_response.content

In [27]:
gr.ChatInterface(bank_expert).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
